# PAD-UFES-20 Colab Training Launcher

Run this notebook in Google Colab with a GPU runtime. It prepares the PAD-UFES-20 dataset, creates patient-safe splits, then launches the reusable image-only baseline, optional hyperparameter sweep, and optional image-plus-metadata multimodal baseline CLIs.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_config(name, default=None):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            return default
    return default


def export_config(name, default=None, required=False):
    value = get_config(name, default)
    if required and not value:
        raise RuntimeError(f'Set {name} in Colab Secrets before training.')
    if value:
        os.environ[name] = str(value)
    return value


REPO_URL = 'https://github.com/SalmaneSossey/mlops-teledermatology.git'
BRANCH = 'main'
REPO_DIR = Path('/content/mlops-teledermatology')
HF_DATASET_REPO = get_config('PAD_UFES20_HF_REPO_ID', 'SalmaneExploring/pad-ufes-20')
DATA_ROOT = Path('/content/pad_ufes_20')
IMAGES_DIR = DATA_ROOT / 'all_images'
IMAGE_BASELINE_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/image_baseline')
MULTIMODAL_OUTPUT_DIR = Path('/content/drive/MyDrive/mlops-teledermatology/runs/multimodal_baseline')
RUN_HPARAM_SWEEP = True
SWEEP_MAX_TRIALS = 8
RETRAIN_BEST_AFTER_SWEEP = True
RUN_MULTIMODAL_BASELINE = True

export_config('DAGSHUB_TOKEN', required=True)
export_config('DAGSHUB_USERNAME')
export_config('DAGSHUB_REPO_OWNER', 'SalmaneSossey')
export_config('DAGSHUB_REPO_NAME', 'mlops-teledermatology')
export_config('DAGSHUB_MLFLOW_TRACKING_URI')

IMAGE_BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MULTIMODAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], cwd='/content', check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())
print('Hugging Face dataset:', HF_DATASET_REPO)
print('PAD-UFES-20 data root:', DATA_ROOT)
print('PAD-UFES-20 images dir:', IMAGES_DIR)
print('Image baseline output dir:', IMAGE_BASELINE_OUTPUT_DIR)
print('Multimodal output dir:', MULTIMODAL_OUTPUT_DIR)
print('MLflow tracking URI:', os.environ.get('DAGSHUB_MLFLOW_TRACKING_URI') or f"https://dagshub.com/{os.environ['DAGSHUB_REPO_OWNER']}/{os.environ['DAGSHUB_REPO_NAME']}.mlflow")


Working directory: /content/mlops-teledermatology
Hugging Face dataset: SalmaneExploring/pad-ufes-20
PAD-UFES-20 data root: /content/pad_ufes_20
PAD-UFES-20 images dir: /content/pad_ufes_20/all_images
Image baseline output dir: /content/drive/MyDrive/mlops-teledermatology/runs/image_baseline
Multimodal output dir: /content/drive/MyDrive/mlops-teledermatology/runs/multimodal_baseline
MLflow tracking URI: https://dagshub.com/SalmaneSossey/mlops-teledermatology.mlflow


In [ ]:
!pip -q install mlflow huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!python -m src.data.download_pad_ufes_20 \
  --repo-id "{HF_DATASET_REPO}" \
  --output-dir "{DATA_ROOT}" \
  --force

!python -m src.data.make_image_splits \
  --metadata-path "{DATA_ROOT / 'metadata.csv'}" \
  --images-dir "{IMAGES_DIR}" \
  --output-dir data/processed/splits


Fetching ... files: 147it [00:30,  6.48it/s]
Fetching ... files: 647it [01:49,  8.04it/s]
Fetching ... files: 1806it [04:03, 18.36it/s]
Fetching ... files: 2153it [04:29, 14.89it/s]
Fetching ... files: 2301it [04:47,  8.01it/s]
Download complete: : 3.57GB [04:54, 12.1MB/s]
Downloaded SalmaneExploring/pad-ufes-20 to /content/pad_ufes_20
Metadata: /content/pad_ufes_20/metadata.csv
Images: /content/pad_ufes_20/all_images
Split distribution by diagnosis:
split       test  train  val
diagnostic                  
ACK          109    511  110
BCC          127    591  127
MEL            8     36    8
NEV           36    171   37
SCC           29    134   29
SEK           35    165   35

Images per split:
split
train    1608
val       346
test      344
Name: count, dtype: int64

Patients per split:
split
train    959
val      208
test     206
Name: patient_id, dtype: int64

Wrote split manifests to data/processed/splits


In [ ]:
!python -m src.training.train_image_baseline \
  --images-dir "{IMAGES_DIR}" \
  --splits-dir data/processed/splits \
  --output-dir "{IMAGE_BASELINE_OUTPUT_DIR}" \
  --hf-dataset-repo "{HF_DATASET_REPO}"


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 155MB/s]
{'epoch': 1, 'seconds': 112.1, 'train_loss': 1.4991138909586625, 'train_macro_f1': 0.34138782640454607, 'train_balanced_acc': 0.4165724953456082, 'train_high_risk_recall': 0.633377135348226, 'val_loss': 1.1714491341155389, 'val_macro_f1': 0.4892353867703867, 'val_balanced_acc': 0.5373799231897249, 'val_high_risk_recall': 0.8414634146341463}
{'epoch': 2, 'seconds': 78.4, 'train_loss': 0.954661871959914, 'train_macro_f1': 0.5682558826644429, 'train_balanced_acc': 0.6356231576045467, 'train_high_risk_recall': 0.7752956636005256, 'val_loss': 1.1383874878028915, 'val_macro_f1': 0.5544924353576062, 'val_balanced_acc': 0.5715588178458121, 'val_high_risk_recall': 0.7987804878048781}
{'epoch': 3, 'seconds': 77.4, 'train_loss': 0.7319458272326645, 'train_macro_f1': 0.6512948443399337, 'train_

## Optional Hyperparameter Sweep

Set `RUN_HPARAM_SWEEP = True` in the setup cell after confirming DagsHub credentials and GPU runtime availability.


In [ ]:
if RUN_HPARAM_SWEEP:
    command = [
        'python', '-m', 'src.training.run_hparam_sweep',
        '--images-dir', str(IMAGES_DIR),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(IMAGE_BASELINE_OUTPUT_DIR),
        '--max-trials', str(SWEEP_MAX_TRIALS),
        '--hf-dataset-repo', HF_DATASET_REPO,
    ]
    if RETRAIN_BEST_AFTER_SWEEP:
        command.append('--retrain-best')
    subprocess.run(command, check=True)
else:
    subprocess.run([
        'python', '-m', 'src.training.run_hparam_sweep',
        '--images-dir', str(IMAGES_DIR),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(IMAGE_BASELINE_OUTPUT_DIR),
        '--max-trials', '3',
        '--dry-run',
    ], check=True)


## Optional Image Plus Metadata Baseline

Set `RUN_MULTIMODAL_BASELINE = True` in the setup cell after reviewing the image-only and metadata-only baselines.


In [ ]:
if RUN_MULTIMODAL_BASELINE:
    subprocess.run([
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--images-dir', str(IMAGES_DIR),
        '--metadata-path', str(DATA_ROOT / 'metadata.csv'),
        '--splits-dir', 'data/processed/splits',
        '--output-dir', str(MULTIMODAL_OUTPUT_DIR),
        '--hf-dataset-repo', HF_DATASET_REPO,
    ], check=True)
else:
    subprocess.run([
        'python', '-m', 'src.training.train_multimodal_baseline',
        '--help',
    ], check=True)
